In [4]:
import kagglehub
path = kagglehub.dataset_download("tongpython/cat-and-dog")

Using Colab cache for faster access to the 'cat-and-dog' dataset.


In [5]:
import os
import cv2
import numpy as np
from tqdm import tqdm

def load_data(data_path):
    images = []
    labels = []
    # 0 for Dog, 1 for Cat
    categories = ['dogs', 'cats']

    for category in categories:
        path = os.path.join(data_path, category)
        class_num = categories.index(category)

        for img in tqdm(os.listdir(path)):
            try:
                img_array = cv2.imread(os.path.join(path, img))
                # Resizing to 224x224 for VGG16
                resized_array = cv2.resize(img_array, (224, 224))
                images.append(resized_array)
                labels.append(class_num)
            except Exception as e:
                pass

    return np.array(images), np.array(labels)



In [14]:
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Define data paths
train_path = os.path.join(path, 'training_set/training_set')
test_path = os.path.join(path, 'test_set/test_set')

# Create ImageDataGenerator instances
# Rescale images to [0, 1] range as part of preprocessing
train_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

# Prepare generators for training and validation data
image_size = (224, 224) # VGG16 input size
batch_size = 16 # Keep batch size consistent with model.fit

train_generator = train_datagen.flow_from_directory(
    train_path,
    target_size=image_size,
    batch_size=batch_size,
    class_mode='binary' # Because we have two classes (cat, dog)
)

test_generator = test_datagen.flow_from_directory(
    test_path,
    target_size=image_size,
    batch_size=batch_size,
    class_mode='binary'
)

print(f'Found {train_generator.samples} training images belonging to {train_generator.num_classes} classes.')
print(f'Found {test_generator.samples} testing images belonging to {test_generator.num_classes} classes.')

Found 8005 images belonging to 2 classes.
Found 2023 images belonging to 2 classes.
Found 8005 training images belonging to 2 classes.
Found 2023 testing images belonging to 2 classes.


In [20]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense , Dropout ,Input , Flatten

base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

model = Sequential([
    base_model,
    Flatten(),
    Dense(128 , activation= 'relu'),
    Dense(1,activation = 'sigmoid')
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

In [21]:
model.fit(
    train_generator,
    epochs=3,
    verbose=1,
    validation_data=test_generator,
    steps_per_epoch=train_generator.samples // train_generator.batch_size,
    validation_steps=test_generator.samples // test_generator.batch_size,
    batch_size = 16
)

Epoch 1/3
500/500 ━━━━━━━━━━━━━━━━━━━━ 73s 141ms/step - accuracy: 0.8735 - loss: 0.3374 - val_accuracy: 0.8963 - val_loss: 0.2762
Epoch 2/3
  1/500 ━━━━━━━━━━━━━━━━━━━━ 54s 109ms/step - accuracy: 0.8750 - loss: 0.2878

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


500/500 ━━━━━━━━━━━━━━━━━━━━ 14s 27ms/step - accuracy: 0.8750 - loss: 0.2878 - val_accuracy: 0.9062 - val_loss: 0.2456
Epoch 3/3
500/500 ━━━━━━━━━━━━━━━━━━━━ 68s 136ms/step - accuracy: 0.9438 - loss: 0.1370 - val_accuracy: 0.9182 - val_loss: 0.2076


In [22]:
test_loss, test_acc = model.evaluate(test_generator, verbose=1)
print(f'Test Accuracy: {test_acc*100:.2f}%')
print(f'Test Loss: {test_loss:.4f}')

127/127 ━━━━━━━━━━━━━━━━━━━━ 18s 144ms/step - accuracy: 0.9184 - loss: 0.2071
Test Accuracy: 91.84%
Test Loss: 0.2071
